In [1]:
import pandas as pd
df=pd.read_csv("../datasets/processed/clean_complaints.csv")

print(df.shape)
print(df.columns)
df.head()

(83349, 12)
Index(['complaint_id', 'date_received', 'product', 'sub_product', 'issue',
       'sub_issue', 'complaint_text', 'company', 'company_response',
       'timely_response', 'complaint_text_clean', 'word_count'],
      dtype='object')


,complaint_id,date_received,product,sub_product,issue,sub_issue,complaint_text,company,company_response,timely_response,complaint_text_clean,word_count
0,7634786,2023-10-03T14:48:38.000Z,Credit card,General-purpose credit card or charge card,Problem with a company's investigation into an...,Was not notified of investigation status or re...,I'm having difficulty accepting this issue and...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",Closed with non-monetary relief,Yes,i m having difficulty accepting this issue and...,38
1,6681064,2023-03-11T15:29:34.000Z,Credit card or prepaid card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Card was charged for something you did not pur...,I called Discover to report my card lost. I to...,DISCOVER BANK,Closed with explanation,Yes,i called discover to report my card lost i tol...,181
2,6681463,2023-03-11T15:35:55.000Z,Credit card or prepaid card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Card was charged for something you did not pur...,My TD Bank Business Credit Card XXXX was last ...,TD BANK US HOLDING COMPANY,Closed with explanation,Yes,my td bank business credit card was last used ...,46
3,6681185,2023-03-11T15:59:56.000Z,Credit card or prepaid card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,On XX/XX/XXXX I ordered a TV online from a ret...,JPMORGAN CHASE & CO.,Closed with explanation,Yes,on i ordered a tv online from a retailer throu...,705
4,7634456,2023-10-03T15:30:39.000Z,Credit card,General-purpose credit card or charge card,Problem with a company's investigation into an...,Was not notified of investigation status or re...,It is completely unjustified that I have consi...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",Closed with non-monetary relief,Yes,it is completely unjustified that i have consi...,31


In [2]:
print("=== ISSUE VALUE COUNTS ===")
print(df['issue'].value_counts().to_string())
print("\n=== SUB-ISSUE VALUE COUNTS ===")  
print(df['sub_issue'].value_counts().head(20).to_string())

=== ISSUE VALUE COUNTS ===
issue
Problem with a purchase shown on your statement                    52340
Problem with a company's investigation into an existing problem     8878
Trouble using your card                                             8842
Problem with a purchase or transfer                                 3106
Billing disputes                                                    3092
Trouble using the card                                              2544
Identity theft / Fraud / Embezzlement                               1710
Customer service / Customer relations                                970
Transaction issue                                                    680
Billing statement                                                    619
Credit monitoring or identity theft protection services              403
Problem with fraud alerts or security freezes                        165

=== SUB-ISSUE VALUE COUNTS ===
sub_issue
Credit card company isn't resolving a dispute abo

In [3]:
def assign_dispute_category(row):
    """
    Assigns one of 6 dispute categories based on issue, sub_issue, and complaint text.
    Uses a priority-ordered rule system — first match wins.
    
    Categories:
      1. Unauthorized Transaction
      2. Billing Error
      3. Merchant Fraud
      4. Goods Not Received
      5. Service Not Provided
      6. Duplicate Charge
    """
    issue     = str(row.get('issue', '')).lower()
    sub_issue = str(row.get('sub_issue', '')).lower()
    text      = str(row.get('complaint_text_clean', 
                    row.get('complaint_text', ''))).lower()

    # ── 1. UNAUTHORIZED TRANSACTION ──────────────────────────────────────
    # Strongest signals first
    unauth_issue_keywords = [
        'fraud', 'unauthorized', 'identity theft', 'embezzlement'
    ]
    unauth_sub_keywords = [
        'did not make', 'unauthorized', 'not authorized', 
        'fraudulent', 'stolen card', 'lost card'
    ]
    unauth_text_keywords = [
        'unauthorized charge', 'did not make this purchase',
        'did not make this transaction', 'i did not authorize',
        'fraudulent charge', 'someone used my card',
        'card was stolen', 'card stolen', 'account hacked',
        'identity theft', 'fraud alert'
    ]
    if (any(k in issue for k in unauth_issue_keywords) or
        any(k in sub_issue for k in unauth_sub_keywords) or
        any(k in text for k in unauth_text_keywords)):
        return 'Unauthorized Transaction'

    # ── 2. DUPLICATE CHARGE ──────────────────────────────────────────────
    dup_keywords = [
        'duplicate', 'charged twice', 'double charge',
        'double billed', 'billed twice', 'charged two times',
        'same charge', 'duplicate transaction'
    ]
    if any(k in text for k in dup_keywords) or 'duplicate' in sub_issue:
        return 'Duplicate Charge'

    # ── 3. MERCHANT FRAUD ────────────────────────────────────────────────
    merchant_keywords = [
        'scam', 'fake merchant', 'fraudulent merchant',
        'merchant fraud', 'fake company', 'never received',
        'fake website', 'fake store', 'deceptive merchant',
        'misleading', 'bait and switch', 'merchant lied',
        'false advertising'
    ]
    if any(k in text for k in merchant_keywords):
        return 'Merchant Fraud'

    # ── 4. GOODS NOT RECEIVED ────────────────────────────────────────────
    goods_issue_keywords = [
        'problem with a purchase or transfer',
        'problem with a purchase shown'
    ]
    goods_sub_keywords = [
        'item not received', 'merchandise received was not as described',
        'not received'
    ]
    goods_text_keywords = [
        'never received', 'never arrived', 'item not received',
        'product not received', 'order never', 'package never',
        'did not receive', 'goods not received', 'never got',
        'not delivered', 'never delivered'
    ]
    if (any(k in sub_issue for k in goods_sub_keywords) or
        any(k in text for k in goods_text_keywords)):
        return 'Goods Not Received'

    # ── 5. BILLING ERROR ─────────────────────────────────────────────────
    billing_issue_keywords = [
        'billing dispute', 'billing', 'overcharged', 
        'problem with a purchase shown on your statement'
    ]
    billing_sub_keywords = [
        'overcharged', 'incorrect amount', 'wrong amount',
        'billing error', 'statement error', 'credit not processed',
        'refund never posted', 'incorrect fee'
    ]
    billing_text_keywords = [
        'wrong amount', 'incorrect charge', 'overcharged',
        'billing error', 'statement is wrong', 'charged wrong',
        'incorrect fee', 'wrong fee', 'charged incorrectly',
        'credit not applied', 'refund not received',
        'refund not posted', 'charged the wrong'
    ]
    if (any(k in issue for k in billing_issue_keywords) or
        any(k in sub_issue for k in billing_sub_keywords) or
        any(k in text for k in billing_text_keywords)):
        return 'Billing Error'

    # ── 6. SERVICE NOT PROVIDED ──────────────────────────────────────────
    service_issue_keywords = [
        'trouble using your card', 'trouble using the card',
        'customer service', 'problem with a company'
    ]
    service_text_keywords = [
        'service not provided', 'service was not rendered',
        'subscription', 'membership', 'cancelled but charged',
        'canceled but still charged', 'charged after cancellation',
        'charged after cancel', 'did not receive service',
        'service never provided', 'ticket', 'reservation',
        'hotel', 'airline', 'gym membership', 'service not rendered'
    ]
    if (any(k in issue for k in service_issue_keywords) or
        any(k in text for k in service_text_keywords)):
        return 'Service Not Provided'

    # ── FALLBACK: use issue field to catch remaining ──────────────────────
    if 'purchase shown on your statement' in issue:
        return 'Billing Error'
    if 'investigation' in issue:
        return 'Billing Error'  # most investigation complaints are billing disputes
    if 'trouble using' in issue:
        return 'Service Not Provided'

    return None  # truly uncategorizable

In [4]:
def get_resolution_category(company_response):
    r = str(company_response).lower()
    if 'monetary relief' in r:
        return 'Resolved'
    if 'in progress' in r:
        return 'Under Investigation'
    # closed with explanation or non-monetary = not resolved
    return 'Not Resolved'

In [5]:
print("Applying labels... (takes ~30 seconds for 80k rows)")

df['transaction_category'] = df.apply(assign_dispute_category, axis=1)
df['resolution_category']  = df['company_response'].apply(get_resolution_category)

print("✅ Done")

Applying labels... (takes ~30 seconds for 80k rows)
✅ Done


In [6]:
total = len(df)
labeled = df['transaction_category'].notna().sum()
unlabeled = df['transaction_category'].isna().sum()

print(f"=== LABELING RESULTS ===")
print(f"Total rows    : {total:,}")
print(f"Labeled       : {labeled:,}  ({labeled/total*100:.1f}%)")
print(f"Unlabeled     : {unlabeled:,}  ({unlabeled/total*100:.1f}%)")

print(f"\n=== CATEGORY DISTRIBUTION ===")
print(df['transaction_category'].value_counts(dropna=False))

print(f"\n=== RESOLUTION DISTRIBUTION ===")
print(df['resolution_category'].value_counts())

=== LABELING RESULTS ===
Total rows    : 83,349
Labeled       : 81,672  (98.0%)
Unlabeled     : 1,677  (2.0%)

=== CATEGORY DISTRIBUTION ===
transaction_category
Billing Error               34220
Service Not Provided        16690
Unauthorized Transaction    16640
Merchant Fraud               9127
Goods Not Received           3435
None                         1677
Duplicate Charge             1560
Name: count, dtype: int64

=== RESOLUTION DISTRIBUTION ===
resolution_category
Not Resolved           55365
Resolved               27983
Under Investigation        1
Name: count, dtype: int64


In [7]:
df_labeled = df[df['transaction_category'].notna()].copy()

print(f"Rows kept for training: {len(df_labeled):,}")

# Check class balance — important for model training
category_counts = df_labeled['transaction_category'].value_counts()
print("\nClass balance:")
for cat, count in category_counts.items():
    pct = count / len(df_labeled) * 100
    bar = '█' * int(pct / 2)
    print(f"  {cat:<30} {count:>6,}  {pct:5.1f}%  {bar}")

# Warn if any class is severely underrepresented
min_count = category_counts.min()
if min_count < 200:
    print(f"\n⚠️  WARNING: '{category_counts.idxmin()}' has only {min_count} samples.")
    print("   Consider merging it with a related category or using oversampling.")

Rows kept for training: 81,672

Class balance:
  Billing Error                  34,220   41.9%  ████████████████████
  Service Not Provided           16,690   20.4%  ██████████
  Unauthorized Transaction       16,640   20.4%  ██████████
  Merchant Fraud                  9,127   11.2%  █████
  Goods Not Received              3,435    4.2%  ██
  Duplicate Charge                1,560    1.9%  


In [9]:
# ── Balance the dataset using undersampling ──────────────────────────────
# Cap the dominant class, oversample the small classes
# Target: no class more than 5x larger than the smallest

from sklearn.utils import resample

TARGET_PER_CLASS = 5000  # enough data, balanced enough for a good model

balanced_parts = []

for category in df_labeled['transaction_category'].unique():
    subset = df_labeled[df_labeled['transaction_category'] == category]
    
    if len(subset) >= TARGET_PER_CLASS:
        # Undersample the large classes
        sampled = resample(subset, 
                           n_samples=TARGET_PER_CLASS, 
                           random_state=42, 
                           replace=False)
    else:
        # Oversample the small classes (duplicate rows with slight variation)
        sampled = resample(subset, 
                           n_samples=TARGET_PER_CLASS, 
                           random_state=42, 
                           replace=True)  # replace=True allows duplication
    
    balanced_parts.append(sampled)

df_balanced = pd.concat(balanced_parts).sample(frac=1, random_state=42).reset_index(drop=True)

print("=== BALANCED DATASET ===")
print(df_balanced['transaction_category'].value_counts())
print(f"\nTotal rows: {len(df_balanced):,}")
print(f"Classes   : {df_balanced['transaction_category'].nunique()}")

=== BALANCED DATASET ===
transaction_category
Service Not Provided        5000
Merchant Fraud              5000
Duplicate Charge            5000
Unauthorized Transaction    5000
Billing Error               5000
Goods Not Received          5000
Name: count, dtype: int64

Total rows: 30,000
Classes   : 6


In [11]:
# Save balanced dataset — this is what the model will train on
save_cols = [
    'complaint_id', 'complaint_text', 'complaint_text_clean',
    'issue', 'sub_issue', 'transaction_category',
    'resolution_category', 'company_response', 'word_count'
]
save_cols = [c for c in save_cols if c in df_balanced.columns]

df_balanced[save_cols].to_csv(
    "../datasets/processed/labeled_complaints.csv", 
    index=False
)

print(f"✅ Balanced dataset saved: labeled_complaints.csv")
print(f"   Rows: {len(df_balanced):,}  (6 categories × ~{TARGET_PER_CLASS:,} each)")

✅ Balanced dataset saved: labeled_complaints.csv
   Rows: 30,000  (6 categories × ~5,000 each)
